# Making Your First Request

This tutorial walks you through the steps necessary to make your first request to SlideRule. By the end of this tutorial you will have used SlideRule to calculate and plot elevations over Grand Mesa, Colorado, using ICESat-2 photon cloud data.

While it is possible to directly access all of ICESat-2 SlideRule's services using any client that communicates over HTTP, it is more practical to use the supplied [Python client](https://github.com/SlideRuleEarth/sliderule), which hides much of the complexity of interacting with each of the services and provides a high-level Python interface for most use-cases.

For this reason, _"using"_ SlideRule is, in practice, the same as writing a Python script that uses the SlideRule Python client package.  The Python client is used to issue science processing requests to `slideruleearth.io` and then analyze the responses that come back.

These processing requests will typically specify a geospatial region of interest (e.g. defined by a GeoJSON file) and instruct the SlideRule system what algorithms it wants to have run on the ICESat-2 data collected within that region.  When SlideRule receives the request, it reads the appropriate source datasets, executes the requested algorithms on that data, and returns the results back to the requesting application.

## Setting Up Your System

__Step 1__: Clone the SlideRule Python examples repository
```bash
$ git clone https://github.com/SlideRuleEarth/sliderule-python.git
```

__Step 2__: Create a conda environment with all the necessary dependencies
```bash
$ cd sliderule-python
$ conda env create -f environment.yml
```

__Step 3__: Activate the sliderule conda environment
```bash
$ conda activate sliderule_env

## Your First Processing Request

Now that you have an environment all setup and ready to use SlideRule, this section will walk you through a very simple example that calculates geolocated elevations in the Grand Mesa region in Colorado at a 20m along-track resolution.

#### Import and initialize the SlideRule Python Client.

In general, the `init` function does not have to be called and the `sliderule` package will use reasonable defaults; but for this example we are turning on _verbose_ log messages so we can get more insight into what is happening.  For a full description of the options available when initializing the `sliderule` package, see the [init](api_reference/sliderule.html#init) documentation.

In [ ]:
from sliderule import sliderule, icesat2
sliderule.init(verbose=True)

#### Create a list of coordinates that represent the Grand Mesa region of interest.

The **grandmesa.geojson** file used in this example is in the same directory as this notebook; alternatively, you can create your own GeoJSON file at [geojson.io](https://geojson.io).

The `toregion` function creates a representation of the geospatial region that is understood by SlideRule.  It accepts both GeoJSON files and Shapefiles.  For a full description of the function, see the [toregion](../../user_guide/icesat2.html#toregion) documentation.

In [ ]:
grand_mesa = sliderule.toregion('grandmesa.geojson')

#### Create a dictionary of processing parameters specifying how the elevations for the region should be calculated.

For a full description of the different processing parameters that are accepted by SlideRule, see [parameters](../../user_guide/icesat2.html#parameters).  The parameters of interest here are _len_ which specifies the total along-track length of the segment used to calculate an elevation, and _res_ which specifies the along-track posting interval of the calculation.

In [ ]:
parms = {
    "poly": grand_mesa["poly"],
    "t0": "2023-01-01",
    "t1": "2024-01-01",
    "srt": icesat2.SRT_LAND,
    "cnf": icesat2.CNF_SURFACE_HIGH,
    "len": 40.0,
    "res": 20.0
}

#### Issue the processing request to SlideRule.

When you hit enter, you should see a scrolling list of log messages saying "*atl06 processing initiated on...*". These messages are normal and expected (and displayed only because of the _verbose_ setting used when we initialized the `sliderule` package).

There are many valid reasons for some resources to return no elements, but most often it is because the resource was identified by NASA's CMR system as crossing the region of interest, yet when SlideRule processed the resource, it did not actually intersect.  This happens because the CMR system adds an off-pointing margin to all ground tracks when calculating intersections and therefore over estimates which resources cross any given region.

When this completes (~30 seconds), the _rsps_ variable should now contain all of the results of the elevations calculated by SlideRule.

In [ ]:
gdf = icesat2.atl06p(parms)

#### Analyze the results using Pandas.

We will display a summary and plot the geolocated elevations returned by SlideRule using measurements collected by ICESat-2 in the Grand Mesa region.
For a full description of all of the fields returned from the `atl06p` function, see the [elevations](../../user_guide/icesat2.html#elevations) documentation.

In [ ]:
gdf.describe()

In [ ]:
import matplotlib.pyplot as plt
gdf.plot(column="h_mean")
plt.show()

Once you've completed this walk-through and are comfortable issuing processing requests to SlideRule, you can take a look at the [Documentation](/) and the other example [Jupyter Notebooks](/getting_started/Examples).